# Preprocess the dataset

We removed irrelevant features as well as features that could potentially introduce future information leakage. Unrealistic values were replaced with NaN for subsequent processing. For example, records with LotSizeSquareFeet = 0 were considered invalid and treated as missing values.

Some features contained a large number of missing values, and directly removing all missing records at the beginning would have resulted in a substantial loss of useful information and a very small number of extreme outliers to reduce their disproportionate influence on regression models. Categorical features were converted using one-hot encoding. For high-cardinality categorical features, only the top 10 most frequent categories were retained, while all remaining categories were grouped into an "Other" category before encoding to avoid generating an excessive number of features.

Finally, the remaining records containing missing values were removed. Missing value imputation was not applied during preprocessing because fitting an imputation strategy on the entire dataset before the train–test split could introduce data leakage. Therefore, removing the remaining missing values was adopted as a safer preprocessing strategy.

In [133]:
import pandas as pd
import glob
import os
import numpy as np
path = r"C:\Users\23035\OneDrive\Desktop\Internship\dataset"
files = glob.glob(os.path.join(path, "*.csv"))
dfs = []
for f in files:
    data_month=pd.read_csv(f)
    date_ym=pd.to_datetime(os.path.basename(f).split('.')[0][-6:],format="%Y%m")
    data_month['date_ym']=date_ym
    dfs.append(data_month)
dataset_total_last6 = pd.concat(dfs)

C:\Users\23035\AppData\Local\Temp\ipykernel_73300\3523058081.py:9: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  data_month=pd.read_csv(f)
C:\Users\23035\AppData\Local\Temp\ipykernel_73300\3523058081.py:9: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  data_month=pd.read_csv(f)


### 1. Handle missing values, invalid values and some outliers




In [134]:
# drop the feature which miss more than 10 % values
dataset_total_last6_drop= dataset_total_last6.loc[:,dataset_total_last6.isna().mean() < 0.2]
dataset_total_last6_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 306330 entries, 0 to 24506
Data columns (total 51 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   BuyerAgentAOR             298023 non-null  object        
 1   ListAgentAOR              306207 non-null  object        
 2   ViewYN                    275444 non-null  object        
 3   PoolPrivateYN             273271 non-null  object        
 4   OriginalListPrice         305427 non-null  float64       
 5   ListingKey                306330 non-null  int64         
 6   ListAgentEmail            305496 non-null  object        
 7   CloseDate                 306330 non-null  object        
 8   ClosePrice                306327 non-null  float64       
 9   ListAgentFirstName        304919 non-null  object        
 10  ListAgentLastName         306311 non-null  object        
 11  Latitude                  306273 non-null  float64       
 12  Longitud

In [135]:
# drop future features and unrelated features
# ClosePrice (label)
# We should use the property's features 
# It should be worked on the property whether it is listed or not
dataset_total_last6_drop = dataset_total_last6_drop.drop(columns=[
    # future features (leakage): like the feature related with list, buyer
    "ListAgentAOR",
    "BuyerAgentAOR",# leakage
    "OriginalListPrice",# leakage
    "CloseDate",# future
    "PurchaseContractDate",# future
    "ContractStatusChangeDate",# future
    "DaysOnMarket",# future
    "ListPrice",
    "ListingContractDate",

    # unrelated features and the feature related with List property
    "ListAgentEmail",
    "ListingKey",
    "ListAgentFirstName",
    "ListAgentLastName",
    "BuyerAgentMlsId",
    "ListAgentEmail",
    "ListingKeyNumeric",
    "UnparsedAddress",
    "ListOfficeName",
    "BuyerOfficeName",
    "ListAgentFullName",
    "BuyerAgentFirstName",
    "BuyerAgentLastName",
    "ListingKeyNumeric",
    "MlsStatus",# all Closed - no info
    "BuyerOfficeAOR",
    "ListingId"
])


In [136]:
dataset_total_last6_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 306330 entries, 0 to 24506
Data columns (total 27 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   ViewYN                 275444 non-null  object        
 1   PoolPrivateYN          273271 non-null  object        
 2   ClosePrice             306327 non-null  float64       
 3   Latitude               306273 non-null  float64       
 4   Longitude              306278 non-null  float64       
 5   PropertyType           306330 non-null  object        
 6   LivingArea             285412 non-null  float64       
 7   MLSAreaMajor           271964 non-null  object        
 8   CountyOrParish         306330 non-null  object        
 9   ParkingTotal           297710 non-null  float64       
 10  PropertySubType        284331 non-null  object        
 11  LotSizeAcres           280567 non-null  float64       
 12  YearBuilt              293895 non-null  float64   

In [138]:
# remove some outliers
upper = dataset_total_last6_drop["ClosePrice"].quantile(0.99)
dataset_total_last6_drop = dataset_total_last6_drop[dataset_total_last6_drop["ClosePrice"] <= upper]

In [139]:
# replace unreasonable value
dataset_total_last6_drop["ClosePrice"] = dataset_total_last6_drop["ClosePrice"].replace(0, np.nan)
dataset_total_last6_drop["LotSizeSquareFeet"] = dataset_total_last6_drop["LotSizeSquareFeet"].replace(0, np.nan)
dataset_total_last6_drop["LotSizeAcres"] = dataset_total_last6_drop["LotSizeAcres"].replace(0, np.nan)
dataset_total_last6_drop["LivingArea"] = dataset_total_last6_drop["LotSizeSquareFeet"].replace(0, np.nan)

In [140]:
# remove duplicate rows
data = dataset_total_last6_drop.drop_duplicates()

In [141]:
# replace some missing value with unknown so that we will not lose too moch important data 
cols = ["PropertySubType","City","CountyOrParish","StateOrProvince","PostalCode","MLSAreaMajor","PropertyType","Levels"]
dataset_total_last6_drop[cols] = dataset_total_last6_drop[cols].fillna("Unknown")

In [142]:
# the correlation between missing value rows
missing = dataset_total_last6_drop.isna().astype(int)
missing_corr = missing.corr()
missing_corr

,ViewYN,PoolPrivateYN,ClosePrice,Latitude,Longitude,PropertyType,LivingArea,MLSAreaMajor,CountyOrParish,ParkingTotal,...,StateOrProvince,FireplaceYN,Stories,Levels,LotSizeArea,NewConstructionYN,GarageSpaces,PostalCode,LotSizeSquareFeet,date_ym
ViewYN,1.000000,0.079434,-0.001662,0.004426,0.001663,NaN,0.030836,NaN,NaN,0.081132,...,NaN,0.300521,0.090283,NaN,-0.008065,0.123447,0.047773,NaN,0.030836,NaN
PoolPrivateYN,0.079434,1.000000,0.000705,0.020079,0.021499,NaN,-0.013468,NaN,NaN,0.496633,...,NaN,0.523172,0.597960,NaN,0.003931,0.104985,0.436902,NaN,-0.013468,NaN
ClosePrice,-0.001662,0.000705,1.000000,-0.000117,-0.000112,NaN,0.000837,NaN,NaN,-0.001486,...,NaN,0.001491,0.001923,NaN,0.000172,0.016654,0.000077,NaN,0.000837,NaN
Latitude,0.004426,0.020079,-0.000117,1.000000,0.953455,NaN,0.012509,NaN,NaN,-0.002298,...,NaN,0.013681,0.018160,NaN,0.014776,0.013930,0.005131,NaN,0.012509,NaN
Longitude,0.001663,0.021499,-0.000112,0.953455,1.000000,NaN,0.013550,NaN,NaN,-0.002191,...,NaN,0.010085,0.016274,NaN,0.015879,0.011120,0.005873,NaN,0.013550,NaN
PropertyType,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LivingArea,0.030836,-0.013468,0.000837,0.012509,0.013550,NaN,1.000000,NaN,NaN,-0.049890,...,NaN,0.109659,0.083926,NaN,0.886866,0.193829,-0.058958,NaN,1.000000,NaN
MLSAreaMajor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CountyOrParish,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ParkingTotal,0.081132,0.496633,-0.001486,-0.002298,-0.002191,NaN,-0.049890,NaN,NaN,1.000000,...,NaN,0.566514,0.368295,NaN,-0.042932,0.065285,0.446338,NaN,-0.049890,NaN


In [143]:
dataset_total_last6_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 303280 entries, 0 to 24506
Data columns (total 27 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   ViewYN                 272654 non-null  object        
 1   PoolPrivateYN          271268 non-null  object        
 2   ClosePrice             303257 non-null  float64       
 3   Latitude               303225 non-null  float64       
 4   Longitude              303230 non-null  float64       
 5   PropertyType           303280 non-null  object        
 6   LivingArea             272519 non-null  float64       
 7   MLSAreaMajor           303280 non-null  object        
 8   CountyOrParish         303280 non-null  object        
 9   ParkingTotal           294698 non-null  float64       
 10  PropertySubType        303280 non-null  object        
 11  LotSizeAcres           271811 non-null  float64       
 12  YearBuilt              290944 non-null  float64   

### 2. Convert datatype

In [144]:
dataset_total_last6_drop["FireplaceYN"].value_counts()

FireplaceYN
True     168952
False    109110
Name: count, dtype: int64

Missing indicator 

In [145]:
tf_map = {True: 1, False: 0, "Unknown":2}

dataset_total_last6_drop["ViewYN_missing"] = dataset_total_last6_drop["ViewYN"].isna().astype(int)
dataset_total_last6_drop["ViewYN"]=dataset_total_last6_drop["ViewYN"].fillna(False).map(tf_map)
dataset_total_last6_drop["FireplaceYN_missing"] = dataset_total_last6_drop["FireplaceYN"].isna().astype(int)
dataset_total_last6_drop["FireplaceYN"]=dataset_total_last6_drop["FireplaceYN"].fillna(False).map(tf_map)
dataset_total_last6_drop["PoolPrivateYN_missing"] = dataset_total_last6_drop["PoolPrivateYN"].isna().astype(int)
dataset_total_last6_drop["PoolPrivateYN"]=dataset_total_last6_drop["PoolPrivateYN"].fillna(False).map(tf_map)
dataset_total_last6_drop["NewConstructionYN_missing"] = dataset_total_last6_drop["NewConstructionYN"].isna().astype(int)
dataset_total_last6_drop["NewConstructionYN"]=dataset_total_last6_drop["NewConstructionYN"].fillna(False).map(tf_map)

C:\Users\23035\AppData\Local\Temp\ipykernel_73300\1749327316.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset_total_last6_drop["ViewYN"]=dataset_total_last6_drop["ViewYN"].fillna(False).map(tf_map)
C:\Users\23035\AppData\Local\Temp\ipykernel_73300\1749327316.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset_total_last6_drop["FireplaceYN"]=dataset_total_last6_drop["FireplaceYN"].fillna(False).map(tf_map)
C:\Users\23035\AppData\Local\Temp\ipykernel_73300\1749327316.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill

In [146]:
dataset_total_last6_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 303280 entries, 0 to 24506
Data columns (total 31 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   ViewYN                     303280 non-null  int64         
 1   PoolPrivateYN              303280 non-null  int64         
 2   ClosePrice                 303257 non-null  float64       
 3   Latitude                   303225 non-null  float64       
 4   Longitude                  303230 non-null  float64       
 5   PropertyType               303280 non-null  object        
 6   LivingArea                 272519 non-null  float64       
 7   MLSAreaMajor               303280 non-null  object        
 8   CountyOrParish             303280 non-null  object        
 9   ParkingTotal               294698 non-null  float64       
 10  PropertySubType            303280 non-null  object        
 11  LotSizeAcres               271811 non-null  float64       

In [147]:
print(dataset_total_last6_drop["PropertyType"].value_counts())
dataset_total_last6_drop = pd.get_dummies(dataset_total_last6_drop, columns=["PropertyType"], drop_first=True)

PropertyType
Residential            202634
ResidentialLease        71928
Land                     9349
ResidentialIncome        8091
ManufacturedInPark       7835
CommercialSale           1768
CommercialLease          1504
BusinessOpportunity       171
Name: count, dtype: int64


In [148]:
dataset_total_last6_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 303280 entries, 0 to 24506
Data columns (total 37 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   ViewYN                           303280 non-null  int64         
 1   PoolPrivateYN                    303280 non-null  int64         
 2   ClosePrice                       303257 non-null  float64       
 3   Latitude                         303225 non-null  float64       
 4   Longitude                        303230 non-null  float64       
 5   LivingArea                       272519 non-null  float64       
 6   MLSAreaMajor                     303280 non-null  object        
 7   CountyOrParish                   303280 non-null  object        
 8   ParkingTotal                     294698 non-null  float64       
 9   PropertySubType                  303280 non-null  object        
 10  LotSizeAcres                     271811 non-null  

In [149]:
dataset_total_last6_drop=dataset_total_last6_drop.dropna()
dataset_total_last6_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 203277 entries, 3 to 24504
Data columns (total 37 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   ViewYN                           203277 non-null  int64         
 1   PoolPrivateYN                    203277 non-null  int64         
 2   ClosePrice                       203277 non-null  float64       
 3   Latitude                         203277 non-null  float64       
 4   Longitude                        203277 non-null  float64       
 5   LivingArea                       203277 non-null  float64       
 6   MLSAreaMajor                     203277 non-null  object        
 7   CountyOrParish                   203277 non-null  object        
 8   ParkingTotal                     203277 non-null  float64       
 9   PropertySubType                  203277 non-null  object        
 10  LotSizeAcres                     203277 non-null  

In [150]:
dataset_total_last6_drop.to_csv("cleaned_dataset.csv", index=False)

# 3. Test train split based on X monthes

— Training set: all months except the most recent complete month.

— Validation set: carve out the second-most-recent complete month from the training set for hyperparameter tuning and model selection — never tune on the test set.

— Test set: the last complete month of available data, touched only once, at the end.

In [151]:
X=2 # past 2 month
months = sorted(dataset_total_last6["date_ym"].unique())
data_train=dataset_total_last6[dataset_total_last6['date_ym'].isin(months[-X:-1])]
data_test=dataset_total_last6[dataset_total_last6['date_ym']==months[-1]]

In [152]:
print(data_train['date_ym'].value_counts())
print(data_test['date_ym'].value_counts())

date_ym
2026-05-01    23260
Name: count, dtype: int64
date_ym
2026-06-01    24507
Name: count, dtype: int64
